In [2]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# 1. Carregar os dados
df = pd.read_csv("moss_em1_dynamic.csv")

# 2. Configurações de layout
datasets = df['dataset'].unique()
n_datasets = len(datasets)
# Criamos uma subfigura por dataset, em uma única coluna
fig = make_subplots(
    rows=n_datasets, cols=1, 
    subplot_titles=[f"<b>{ds}</b>" for ds in datasets],
    vertical_spacing=0.02 # Espaço curto entre os gráficos
)

# 3. Iterar e adicionar cada gráfico com sua própria ordem
for i, ds in enumerate(datasets, 1):
    df_ds = df[df['dataset'] == ds].copy()
    
    # Calcular a ordem local (pela mediana do erro neste dataset)
    ordem_local = df_ds.groupby("modelo")["erro"].median().sort_values().index.tolist()
    
    # Adicionar um boxplot para cada modelo, seguindo a ordem local
    for modelo in ordem_local:
        df_mod = df_ds[df_ds['modelo'] == modelo]
        fig.add_trace(
            go.Box(
                y=df_mod['erro'],
                name=modelo,
                boxpoints='outliers',
                legendgroup=modelo,
                showlegend=(i == 1) # Só mostra a legenda no primeiro gráfico
            ),
            row=i, col=1
        )

# 4. Ajustes finais de tamanho e estética
fig.update_layout(
    height=n_datasets * 400, # 300px para cada dataset
    template="plotly_white",
    title_text="Performance Local: Modelos Ordenados do Melhor para o Pior por Dataset",
    margin=dict(t=100, b=50, l=50, r=50),
    showlegend=True
)

# Deixar os eixos X independentes para cada subgráfico respeitar sua ordem
fig.update_xaxes(showgrid=False)
fig.update_yaxes(title_text="MAE")
#fig.write_html("meu_resultado_ordenado.html")
fig.show()

In [3]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# =========================
# 1. Carregar os dados
# =========================
df_mn = pd.read_csv("emqm_30_bcts.csv")
df_d  = pd.read_csv("../exp_011/d_30_bcts.csv")

# =========================
# 2. Filtrar apenas MoSS
# =========================
df_mn["modelo"] = df_mn["modelo"].astype(str)
df_d["modelo"]  = df_d["modelo"].astype(str)

df_mn = df_mn[df_mn["modelo"].str.startswith("MoSS_")].copy()
df_d  = df_d[df_d["modelo"].str.startswith("MoSS_")].copy()

df_mn["origem"] = "MN"
df_d["origem"]  = "D"

# =========================
# 3. Unir os dados
# =========================
df = pd.concat([df_mn, df_d], ignore_index=True)

datasets = df["dataset"].unique()
n_datasets = len(datasets)

if n_datasets == 0:
    raise ValueError("Nenhum dataset encontrado após o filtro de MoSS.")

# =========================
# 4. Criar subplots
# =========================
fig = make_subplots(
    rows=n_datasets,
    cols=1,
    subplot_titles=[f"<b>{ds}</b>" for ds in datasets],
    vertical_spacing=0.03
)

# =========================
# 5. Plotar
# =========================
for i, ds in enumerate(datasets, 1):
    df_ds = df[df["dataset"] == ds].copy()

    # Ordem dos MODELOS (melhor → pior)
    ordem_modelos = (
        df_ds.groupby("modelo")["erro"]
        .median()
        .sort_values()
        .index
        .tolist()
    )

    x_order = []

    for modelo in ordem_modelos:
        df_mod = df_ds[df_ds["modelo"] == modelo]

        med_mn = df_mod[df_mod["origem"] == "MN"]["erro"].median()
        med_d  = df_mod[df_mod["origem"] == "D"]["erro"].median()

        # 🔥 decidir quem fica à esquerda
        if med_mn <= med_d:
            origens_ordenadas = ["MN", "D"]
        else:
            origens_ordenadas = ["D", "MN"]

        # registrar ordem no eixo X
        for origem in origens_ordenadas:
            x_order.append(f"{modelo} ({origem})")

            df_plot = df_mod[df_mod["origem"] == origem]
            if df_plot.empty:
                continue

            fig.add_trace(
                go.Box(
                    y=df_plot["erro"],
                    name=f"{modelo} ({origem})",
                    legendgroup=f"{modelo}_{origem}",
                    boxpoints="outliers",
                    showlegend=(i == 1)
                ),
                row=i,
                col=1
            )

    # 🔒 Forçar ordem correta no eixo X
    fig.update_xaxes(
        categoryorder="array",
        categoryarray=x_order,
        row=i,
        col=1
    )


# =========================
# 6. Layout final
# =========================
fig.update_layout(
    height=n_datasets * 450,
    template="plotly_white",
    title_text="Comparação MN × D — MoSS (30 BCTs por classe)",
    margin=dict(t=100, b=50, l=50, r=50),
    showlegend=True
)

fig.update_yaxes(title_text="MAE")
fig.update_xaxes(showgrid=False)

fig.show()

FileNotFoundError: [Errno 2] No such file or directory: 'emqm_30_bcts.csv'

In [ ]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# =========================
# 1. Carregar os dados
# =========================
df_mn = pd.read_csv("mn_30_bcts.csv")
df_d  = pd.read_csv("../exp_011/d_30_bcts.csv")

# =========================
# 2. Filtrar apenas MoSS
# =========================
df_mn["modelo"] = df_mn["modelo"].astype(str)
df_d["modelo"]  = df_d["modelo"].astype(str)

df_mn = df_mn[df_mn["modelo"].str.startswith("MoSS_")].copy()
df_d  = df_d[df_d["modelo"].str.startswith("MoSS_")].copy()

df_mn["origem"] = "MN"
df_d["origem"]  = "D"

# =========================
# 3. Unir os dados
# =========================
df = pd.concat([df_mn, df_d], ignore_index=True)

datasets = df["dataset"].unique()
n_datasets = len(datasets)

if n_datasets == 0:
    raise ValueError("Nenhum dataset encontrado após o filtro de MoSS.")

# =========================
# 4. Criar subplots
# =========================
fig = make_subplots(
    rows=n_datasets,
    cols=1,
    subplot_titles=[f"<b>{ds}</b>" for ds in datasets],
    vertical_spacing=0.03
)

# =========================
# 5. Plotar
# =========================
for i, ds in enumerate(datasets, 1):
    df_ds = df[df["dataset"] == ds].copy()

    # Ordem dos MODELOS (melhor → pior)
    ordem_modelos = (
        df_ds.groupby("modelo")["erro"]
        .median()
        .sort_values()
        .index
        .tolist()
    )

    x_order = []

    for modelo in ordem_modelos:
        df_mod = df_ds[df_ds["modelo"] == modelo]

        med_mn = df_mod[df_mod["origem"] == "MN"]["erro"].median()
        med_d  = df_mod[df_mod["origem"] == "D"]["erro"].median()

        # 🔥 decidir quem fica à esquerda
        if med_mn <= med_d:
            origens_ordenadas = ["MN", "D"]
        else:
            origens_ordenadas = ["D", "MN"]

        # registrar ordem no eixo X
        for origem in origens_ordenadas:
            x_order.append(f"{modelo} ({origem})")

            df_plot = df_mod[df_mod["origem"] == origem]
            if df_plot.empty:
                continue

            fig.add_trace(
                go.Box(
                    y=df_plot["erro"],
                    name=f"{modelo} ({origem})",
                    legendgroup=f"{modelo}_{origem}",
                    boxpoints="outliers",
                    showlegend=(i == 1)
                ),
                row=i,
                col=1
            )

    # 🔒 Forçar ordem correta no eixo X
    fig.update_xaxes(
        categoryorder="array",
        categoryarray=x_order,
        row=i,
        col=1
    )


# =========================
# 6. Layout final
# =========================
fig.update_layout(
    height=n_datasets * 450,
    template="plotly_white",
    title_text="Comparação MN × D — MoSS (30 BCTs por classe)",
    margin=dict(t=100, b=50, l=50, r=50),
    showlegend=True
)

fig.update_yaxes(title_text="MAE")
fig.update_xaxes(showgrid=False)

fig.show()

FileNotFoundError: [Errno 2] No such file or directory: 'mn_30_bcts.csv'

In [ ]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# 1. Carregar os dados
df = pd.read_csv("mn_30_bcts.csv")

# 2. Configurações de layout
datasets = df['dataset'].unique()
n_datasets = len(datasets)
# Criamos uma subfigura por dataset, em uma única coluna
fig = make_subplots(
    rows=n_datasets, cols=1, 
    subplot_titles=[f"<b>{ds}</b>" for ds in datasets],
    vertical_spacing=0.02 # Espaço curto entre os gráficos
)

# 3. Iterar e adicionar cada gráfico com sua própria ordem
for i, ds in enumerate(datasets, 1):
    df_ds = df[df['dataset'] == ds].copy()
    
    # Calcular a ordem local (pela mediana do erro neste dataset)
    ordem_local = df_ds.groupby("modelo")["erro"].median().sort_values().index.tolist()
    
    # Adicionar um boxplot para cada modelo, seguindo a ordem local
    for modelo in ordem_local:
        df_mod = df_ds[df_ds['modelo'] == modelo]
        fig.add_trace(
            go.Box(
                y=df_mod['erro'],
                name=modelo,
                boxpoints='outliers',
                legendgroup=modelo,
                showlegend=(i == 1) # Só mostra a legenda no primeiro gráfico
            ),
            row=i, col=1
        )

# 4. Ajustes finais de tamanho e estética
fig.update_layout(
    height=n_datasets * 400, # 300px para cada dataset
    template="plotly_white",
    title_text="Performance Local: Modelos Ordenados do Melhor para o Pior por Dataset",
    margin=dict(t=100, b=50, l=50, r=50),
    showlegend=True
)

# Deixar os eixos X independentes para cada subgráfico respeitar sua ordem
fig.update_xaxes(showgrid=False)
fig.update_yaxes(title_text="MAE")
#fig.write_html("meu_resultado_ordenado.html")
fig.show()